## Here is an example of creating SQL Lite database

Additional resources:
https://www.geeksforgeeks.org/python-sqlite/

In [ ]:
import pandas as pd
import sqlite3

In [ ]:
# Read the Excel file
df = pd.read_csv('data/Online Retail_short.csv')
df.dropna(inplace=True)

# Data for this example is available here: 
# https://archive.ics.uci.edu/static/public/352/online+retail.zip


In [ ]:
df.columns

In [ ]:
df.head()

### Preapering data for database tables

In [ ]:
# Prepare DataFrames for Customers, Products, and Transactions tables
customers_df = df[['CustomerID', 'Country']].drop_duplicates(subset=['CustomerID']).astype({'CustomerID': 'int'}).reset_index(drop=True)
products_df = df[['StockCode', 'Description']].drop_duplicates(subset=['StockCode']).astype({'StockCode': 'str'}).reset_index(drop=True)
transactions_df = df[['InvoiceNo', 'StockCode', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID']].copy()
transactions_df['InvoiceDate'] = transactions_df['InvoiceDate'].astype(str) 

In [ ]:
# Create SQLite Database and cursor
conn = sqlite3.connect('ECommerceDB.sqlite')
cursor = conn.cursor()

In [ ]:
# Enable foreign key support
cursor.execute("PRAGMA foreign_keys = ON;")

In [ ]:
# Create Customers table with a primary key
cursor.execute('''
CREATE TABLE IF NOT EXISTS Customers (
    CustomerID INTEGER PRIMARY KEY,
    Country TEXT
)
''')

In [ ]:
# Create Products table with a primary key
cursor.execute('''
CREATE TABLE IF NOT EXISTS Products (
    StockCode TEXT PRIMARY KEY,
    Description TEXT
)
''')

In [ ]:
# Create Transactions table with primary key and foreign keys
cursor.execute('''
CREATE TABLE IF NOT EXISTS Transactions (
    TransactionID INTEGER PRIMARY KEY AUTOINCREMENT,
    InvoiceNo TEXT,
    InvoiceDate TEXT,
    Quantity INTEGER,
    UnitPrice REAL,
    StockCode TEXT,
    CustomerID INTEGER,
    FOREIGN KEY (StockCode) REFERENCES Products(StockCode),
    FOREIGN KEY (CustomerID) REFERENCES Customers(CustomerID)
)
''')

In [ ]:
# Insert Data into Tables
customers_df.to_sql('Customers', conn, if_exists='append', index=False)
products_df.to_sql('Products', conn, if_exists='append', index=False)
transactions_df.to_sql('Transactions', conn, if_exists='append', index=False, chunksize=200)  # Adjust chunksize as necessary

In [ ]:
# Close Connection
conn.close()

### We do example search 

Search 1: Find all the information about transaction InvoiceNo 536365

Search 2: Find customers (top 5) with the highest number of purchases (distinct InvoiceNo)

Search 3: Find the most expensive products

In [ ]:
# Search 1

conn = sqlite3.connect('ECommerceDB.sqlite')
print(pd.read_sql_query("SELECT * FROM Transactions WHERE InvoiceNo = 536365;", conn))
conn.close()

In [ ]:
# Search 2
conn = sqlite3.connect('ECommerceDB.sqlite')
query = '''
    SELECT 
        Transactions.CustomerID, 
        COUNT(DISTINCT Transactions.InvoiceNo) AS PurchaseCount
    FROM 
        Transactions
    GROUP BY 
        Transactions.CustomerID
    ORDER BY 
        PurchaseCount DESC
    LIMIT 5;
'''
result = pd.read_sql_query(query, conn)

conn.close()
print(result)

In [ ]:
# Search 3
conn = sqlite3.connect('ECommerceDB.sqlite')
query = '''
SELECT 
    p.StockCode,
    p.Description,
    MAX(t.UnitPrice) AS MaxPrice
FROM 
    Transactions t
JOIN 
    Products p ON t.StockCode = p.StockCode
GROUP BY 
    p.StockCode, p.Description
ORDER BY 
    MaxPrice DESC
LIMIT 10;

'''
result = pd.read_sql_query(query, conn)

conn.close()
print(result)